# Evaluación de Tavily como search tool de un agente LangGraph

## 1. Planteamiento

**Contexto.** La empresa quiere implementar un chatbot para consultas web. Se me encarga evaluar **Tavily** como `search_tool`.

**Objetivo.** Entender cómo se comporta Tavily dentro de un agente, cuánto cuesta en tokens y si se puede mejorar su coste y su relevancia.

**Plan de trabajo**

| Paso | Contenido |
|---|---|
| 1 | Comprobar que el agente busca cuando debe y solo entonces · acceder a los `ToolMessage` · medir `token_usage` y el peso de los `ToolMessage` |
| 2 | Entender el `search_tool`: estructura, snippets, score, parámetros, schema de `bind_tools()`, ¿el modelo usa los resultados o alucina? |
| 3 | Mejorar coste y relevancia: filtrar por score, no inyectar búsquedas infructuosas, prompt caching, relevancia evaluada por el LLM |

**Decisiones de diseño**

- Modelo: `claude-haiku-4-5` (corte de conocimiento: julio 2025).
- En vez de un *tool* para la fecha, el **system prompt** inyecta el corte de conocimiento y la fecha de hoy. Así el modelo sabe cuándo debe buscar sin gastar una llamada a una herramienta.
- La lógica está en `src/victorg_langgraph_websearchagent/agent_v2.py`; este notebook solo la usa y muestra resultados.

## 2. Preparación

In [1]:
%load_ext autoreload
%autoreload 2

import json

from victorg_langgraph_websearchagent.agent_v2 import (
    load_keys,
    build_tools,
    build_search_agent,
    run_agent,
    get_answer,
    used_search,
    print_summary,
    get_tool_messages,
    print_tool_messages,
    count_tokens,
    print_tokens,
    toolmessage_tokens,
    print_toolmessage_weight,
    verify_toolmessage_tokens,
    print_weight_summary,
    # Paso 2
    get_tavily_outputs,
    print_tavily_structure,
    print_tool_schemas,
    print_tavily_parameters,
    print_tool_schema_tokens,
    first_call_input,
    inspect_snippets,
    print_snippet_report,
    judge_results_relevance,
    print_score_vs_relevance,
    evaluate_groundedness,
    print_groundedness,
)

load_keys()
agent = build_search_agent()


## 3. Paso 1: comportamiento, acceso a ToolMessages y métricas de tokens

### 3.1 ¿El agente ejecuta la tool cuando debe (y solo entonces)?

Cada query lleva el comportamiento esperado: `expected_search=False` (conocimiento estático o fecha ya inyectada) o `True` (información actual).
Incluye casos límite cerca del corte (julio 2025) y una pregunta que comprueba la inyección de fecha (`¿Qué día es hoy?`, que **no** debe buscar).

In [2]:
queries = [
    # --- Estático / atemporal: NO debe buscar ---
    {"query": "¿Quién escribió Don Quijote de la Mancha?", "expected_search": False},
    {"query": "Explica qué es el teorema de Pitágoras", "expected_search": False},
    {"query": "¿Quién ganó el Mundial de fútbol de Qatar 2022?", "expected_search": False},
    {"query": "¿Qué es una lista por comprensión en Python?", "expected_search": False},
    {"query": "Hola, ¿qué puedes hacer?", "expected_search": False},
    # --- Fecha: está inyectada en el system prompt, NO debe buscar ---
    {"query": "¿Qué día es hoy?", "expected_search": False},
    # --- Herramienta multiply: NO debe buscar ---
    {"query": "Multiplica 123 por 456", "expected_search": False},
    # --- Información actual: DEBE buscar ---
    {"query": "¿Cuál es la última versión estable de LangGraph?", "expected_search": True},
    {"query": "¿Qué tiempo hace hoy en Madrid?", "expected_search": True},
    {"query": "¿Cuál es el precio actual del Bitcoin?", "expected_search": True},
    {"query": "Resume las noticias de hoy sobre inteligencia artificial", "expected_search": True},
    # --- Casos límite cerca del corte (julio 2025) ---
    {"query": "¿Quién ganó la Champions League 2024-2025?", "expected_search": False},
    {"query": "¿Quién ganó el Mundial de Clubes de la FIFA 2025?", "expected_search": True},
    {"query": "¿Quién ganó el Tour de Francia 2025?", "expected_search": True},
    # --- Mezcla: multiply + web ---
    {"query": "Multiplica 12 por 34 y dime cuál es la última versión estable de LangGraph", "expected_search": True},
]

In [3]:
results = [run_agent(agent, q["query"]) for q in queries]

In [4]:
# ¿Usó la web cuando debía? (✓ = coincide con lo esperado)
print_summary(queries, results)

#   Query                                                Esperado  Usó web   Tokens   Tool  OK
1   ¿Quién escribió Don Quijote de la Mancha?            estático  no          2701      0  ✓
2   Explica qué es el teorema de Pitágoras               estático  no          3017      0  ✓
3   ¿Quién ganó el Mundial de fútbol de Qatar 2022?      estático  no          2694      0  ✓
4   ¿Qué es una lista por comprensión en Python?         estático  no          3076      0  ✓
5   Hola, ¿qué puedes hacer?                             estático  no          2897      0  ✓
6   ¿Qué día es hoy?                                     estático  no          2583      0  ✓
7   Multiplica 123 por 456                               estático  no          5283     83  ✓
8   ¿Cuál es la última versión estable de LangGraph?     web       sí          7985   2522  ✓
9   ¿Qué tiempo hace hoy en Madrid?                      web       sí         10274   4790  ✓
10  ¿Cuál es el precio actual del Bitcoin?               we

### 3.2 Acceso a los ToolMessages

Un `ToolMessage` es el mensaje donde LangGraph guarda lo que devuelve una herramienta. Vive en `result["messages"]`.
`get_tool_messages(result)` los devuelve en una lista y `print_tool_messages(result)` los muestra legibles.

In [5]:
i = 7  # "¿Cuál es la última versión estable de LangGraph?"
print(queries[i]["query"], "\n")

# Acceso directo: lista de ToolMessage y contenido como diccionario
tool_messages = get_tool_messages(results[i])
tm = tool_messages[0]
tavily_output = json.loads(tm.content)

print("Herramienta:", tm.name)
print("tool_call_id:", tm.tool_call_id)
print("Claves de la salida de Tavily:", list(tavily_output.keys()))

¿Cuál es la última versión estable de LangGraph? 

Herramienta: tavily_search
tool_call_id: toolu_014HzVtYVTYHr86mkCrCXhzg
Claves de la salida de Tavily: ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']


In [6]:
# Vista legible de todos los ToolMessages de esa ejecución
print_tool_messages(results[i])

ToolMessage 1: tavily_search
Argumentos: {'query': 'LangGraph última versión estable 2026', 'search_depth': 'basic'}
----------------------------------------------------------------------
Query: LangGraph última versión estable 2026
Tiempo de respuesta: 1.45 s
Nº de resultados: 5

[1] LangChain vs LangGraph (2026): When to Use Which
    URL:   https://www.ayautomate.com/blog/langgraph-vs-langchain
    Score: 0.71611935
    Texto: LangGraph hit v1.0 on October 22, 2025, alongside LangChain 1.0. Current version: 1.2.7 (released June 30, 2026). Notable 2026 additions: DeltaChannel (beta), which stores only incremental state deltas rather than full snapshots on each checkpoint write, reducing storage overhead for large-state gra...

[2] @langchain/langgraph - npm
    URL:   https://www.npmjs.com/package/@langchain/langgraph
    Score: 0.45577386
    Texto: ⚠️  npm tokens that bypass 2FA are being restricted — account changes (Aug 2026) and direct publishing (Jan 2027). Learn how to prepare

### 3.3 Métricas de `token_usage` y peso de los ToolMessages

**Qué da la API.** Cada respuesta del modelo trae `usage_metadata` con `input_tokens`, `output_tokens` y `total_tokens` **de esa llamada**. La API **no** dice cuántos tokens tiene cada mensaje por separado. Hay que deducirlo.

**Cómo se determinan los tokens de un ToolMessage (no de un tool_call).**

En un agente con una herramienta, el modelo se llama dos veces. En cada llamada el modelo recibe **todo el contexto acumulado**:

```
Llamada 1   input I1 = system prompt + schemas de las tools + pregunta
            output O1 = el tool_call (el modelo pide la herramienta)

Llamada 2   input I2 = system prompt + schemas de las tools + pregunta
                       + el tool_call (el AIMessage de la llamada 1)
                       + el ToolMessage (resultado de la herramienta)
```

Restando una llamada a la otra, todo lo que se repite (system prompt, schemas, pregunta) se **cancela**:

```
I2 - I1 = tokens del tool_call (como input) + tokens del ToolMessage
```

El tool_call es lo que el modelo **generó** en la llamada 1, y su tamaño lo conocemos: es `O1`. Al volver a entrar como contexto ocupa esos mismos tokens. Por tanto:

```
tokens del ToolMessage = I2 - I1 - O1
```

**Diferencia entre tool_call y ToolMessage**

| | tool_call | ToolMessage |
|---|---|---|
| Quién lo escribe | el modelo | la herramienta (Tavily) |
| Cómo se factura | **output** la primera vez (`O1`) y luego input | solo **input** |
| Cómo se mide | directamente: `output_tokens` de la llamada 1 | por diferencia: `I2 - I1 - O1` |

**Dos matices importantes**

- **Se factura en cada llamada posterior.** El ToolMessage queda en el contexto. Si después de él el modelo se llama 2 veces más, esos tokens se pagan 2 veces como input. Por eso se calcula también el `input_facturado = tokens × nº de llamadas que lo arrastran`.
- **Es una deducción, no una medida directa.** Incluye unos pocos tokens de estructura (`tool_use` / `tool_result`) y puede haber una diferencia mínima al volver a codificar el tool_call. Para comprobarlo, `verify_toolmessage_tokens` cuenta con el endpoint `count_tokens` de Anthropic el texto puro del ToolMessage y lo compara.

**Peso del ToolMessage** = tokens de ToolMessage facturados / input total (o / tokens totales). El coste usa el precio de entrada de Haiku 4.5 (1 USD por millón de tokens).

In [7]:
# Datos crudos: usage_metadata de cada llamada al modelo
for m in results[i]["messages"]:
    usage = getattr(m, "usage_metadata", None)
    if usage:
        print(usage)

{'input_tokens': 2564, 'output_tokens': 84, 'total_tokens': 2648, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}
{'input_tokens': 5086, 'output_tokens': 251, 'total_tokens': 5337, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}


In [8]:
# Desglose por llamada + tokens del tool_call
print_tokens(results[i])

Llamada 1 (tavily_search): input=2564 | output=84 | total=2648
Llamada 2 (respuesta final): input=5086 | output=251 | total=5337
TOTAL: input=7650 | output=335 | total=7985
TOOL CALL: petición=84 | resultado=2438 | total herramienta=2522


In [9]:
# Tokens del ToolMessage con la cuenta paso a paso (I2 - I1 - O1)
print_toolmessage_weight(results[i])

Ronda 1 (tavily_search)
  I1 (input antes del tool_call)      = 2564
  O1 (output: el tool_call)           = 84
  I2 (input con el ToolMessage)       = 5086
  ToolMessage = I2 - I1 - O1          = 2438 tokens (48% del input de la llamada siguiente)
  Llamadas que lo arrastran: 1 -> input facturado por el ToolMessage = 2438
TOTAL ToolMessages: 2438 tokens | 32% del input total | 31% de los tokens totales | coste ~ $0.00244


In [10]:
# Comprobación independiente con el endpoint count_tokens de Anthropic
verify_toolmessage_tokens(results[i])

Ronda 1: por diferencia (I2-I1-O1) = 2438 | texto del ToolMessage con count_tokens = 2426 | diferencia = 12
(la diferencia son los tokens de la estructura tool_use/tool_result, pocos y constantes)


In [11]:
# Peso de los ToolMessages en todas las queries que usaron herramientas
print_weight_summary(queries, results)

#   Query                                           TM tok  Facturado  % input       USD
7   Multiplica 123 por 456                              14         14       0%   0.00001
8   ¿Cuál es la última versión estable de Lan...      2438       2438      32%   0.00244
9   ¿Qué tiempo hace hoy en Madrid?                   4713       4713      48%   0.00471
10  ¿Cuál es el precio actual del Bitcoin?            2798       2798      35%   0.00280
11  Resume las noticias de hoy sobre intelige...      2005       2005      28%   0.00200
12  ¿Quién ganó la Champions League 2024-2025?        3018       3018      37%   0.00302
13  ¿Quién ganó el Mundial de Clubes de la FI...      2963       2963      36%   0.00296
14  ¿Quién ganó el Tour de Francia 2025?              3018       3018      37%   0.00302
15  Multiplica 12 por 34 y dime cuál es la úl...      2407       2407      31%   0.00241


In [12]:
# Medias (solo queries con búsqueda web)
weights = [toolmessage_tokens(r) for r in results]
searched_idx = [k for k, w in enumerate(weights) if w["rondas"]]

if searched_idx:
    avg_tokens = sum(weights[k]["toolmessage_tokens"] for k in searched_idx) / len(searched_idx)
    avg_weight = sum(weights[k]["peso_en_input_total"] for k in searched_idx) / len(searched_idx)
    avg_cost = sum(weights[k]["coste_usd"] for k in searched_idx) / len(searched_idx)
    print(f"Queries con herramientas: {len(searched_idx)} de {len(queries)}")
    print(f"Tokens medios de ToolMessage: {avg_tokens:.0f}")
    print(f"Peso medio en el input total: {avg_weight:.0%}")
    print(f"Coste medio de los ToolMessages: ${avg_cost:.5f}")

Queries con herramientas: 9 de 15
Tokens medios de ToolMessage: 2597
Peso medio en el input total: 31%
Coste medio de los ToolMessages: $0.00260


## 4. Paso 2: entender y evaluar la search tool

### Conceptos antes de empezar

- **search_tool**: una función que el LLM puede *pedir* que se ejecute. El LLM no ejecuta nada: solo ve su "ficha" (nombre, descripción y parámetros) y decide si la llama y con qué argumentos.
- **`bind_tools()`**: convierte cada tool en un **schema JSON** (nombre, descripción y cada parámetro con su descripción). Ese schema se envía **en cada llamada al modelo**, por eso ocupa tokens fijos aunque no se use la tool.
- **Tavily**: motor de búsqueda para agentes. Recibe una query (y parámetros opcionales) y devuelve resultados ordenados por relevancia, cada uno con `title`, `url`, `content` (el *snippet*) y `score`.
- **Snippet (`content`)**: según la [documentación de Tavily](https://docs.tavily.com/api-reference/endpoint/search), son *chunks* cortos extraídos **tal cual de la página** (unos 500 caracteres en `basic`/`fast`, 800 en `advanced`), hasta `chunks_per_source` por URL (3 por defecto), unidos por `[...]`. La profundidad `ultra-fast` devuelve en cambio un resumen NLP por URL.
- **Score**: puntuación de relevancia (decimal) entre la query y el contenido. La documentación dice que cuanto más alto mejor y que el umbral ideal depende del caso de uso, pero **no explica cómo se calcula**. Lo trataremos como una caja negra y lo contrastaremos con un juicio independiente.

En este paso se responden siete preguntas, una por sección.

### 4.1 ¿Qué devuelve Tavily y cuál es la estructura del mensaje?

Lo que devuelve Tavily viaja en el `ToolMessage` como JSON. Abajo se ven sus claves reales, el tamaño de cada campo y los resultados en el orden en que llegan.

In [13]:
# Primera query del Paso 1 que usó la búsqueda web
i_web = next(k for k, r in enumerate(results) if get_tavily_outputs(r))
print(queries[i_web]["query"], "\n")

tavily_outputs = get_tavily_outputs(results[i_web])   # una salida por búsqueda
print_tavily_structure(tavily_outputs[0])

¿Cuál es la última versión estable de LangGraph? 

Claves de nivel superior: ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']
Nº de resultados: 5 | response_time: 1.45
Campos de cada resultado: ['url', 'title', 'content', 'score', 'raw_content', 'id']
Tamaño del JSON: 6954 caracteres (~1738 tokens, orientativo)

Peso por campo (suma de todos los resultados):
  content        5508 caracteres  (79% del JSON)
  url             298 caracteres  (4% del JSON)
  title           242 caracteres  (3% del JSON)
  score            48 caracteres  (1% del JSON)
  id               45 caracteres  (1% del JSON)
  raw_content      20 caracteres  (0% del JSON)

Resultados (ordenados como los devuelve Tavily):
  1. score=0.71611935 | content=1407 caracteres | LangChain vs LangGraph (2026): When to Use Which
     https://www.ayautomate.com/blog/langgraph-vs-langchain
  2. score=0.45577386 | content=1255 caracteres | @langchain/langgraph - npm
     https://www.n

### 4.2 ¿Qué parámetros y qué schema ve el LLM con `bind_tools()`?

Aquí están los ~2000 tokens que se inyectan en el primer input. Se muestra:

1. Los schemas tal y como los recibe el modelo.
2. Qué parámetros decide el LLM y cuáles están fijados al crear la tool (si un parámetro se fija al crear `TavilySearch`, el valor que ponga el LLM **se ignora** aunque siga viendo el parámetro y pagando sus tokens).
3. Los tokens que suman los schemas, medidos con el endpoint `count_tokens` de Anthropic (total, por tool y parte fija).

In [14]:
tools = build_tools()
search_tool = tools[1]   # TavilySearch

print_tool_schemas(tools)

Herramienta: multiply
Descripción: 75 caracteres
Parámetros que ve el LLM: 2
  - a                integer        obligatorio  descripción: 0 caracteres
  - b                integer        obligatorio  descripción: 0 caracteres
JSON del schema: 246 caracteres
Herramienta: tavily_search
Descripción: 375 caracteres
Parámetros que ve el LLM: 9
  - query            string         obligatorio  descripción: 23 caracteres
  - include_domains  array          opcional     descripción: 622 caracteres
  - exclude_domains  array          opcional     descripción: 651 caracteres
  - search_depth     string         opcional     descripción: 443 caracteres
  - include_images   boolean        opcional     descripción: 404 caracteres
  - time_range       string         opcional     descripción: 468 caracteres
  - topic            string         opcional     descripción: 471 caracteres
  - start_date       string         opcional     descripción: 759 caracteres
  - end_date         string         opciona

In [15]:
print_tavily_parameters(search_tool)

Parámetros que VE el LLM:
  - query            lo escribe el LLM
  - include_domains  lo decide el LLM en cada llamada
  - exclude_domains  lo decide el LLM en cada llamada
  - search_depth     lo decide el LLM en cada llamada
  - include_images   lo decide el LLM en cada llamada
  - time_range       lo decide el LLM en cada llamada
  - topic            fijado al crear la tool = 'general' -> el valor del LLM se IGNORA
  - start_date       lo decide el LLM en cada llamada
  - end_date         lo decide el LLM en cada llamada

Parámetros que NO ve el LLM (solo al crear la tool):
  - max_results                  = 5
  - include_answer               = None
  - include_raw_content          = None
  - include_image_descriptions   = None
  - country                      = None
  - include_favicon              = None
  - include_usage                = None
  - exact_match                  = None
  - auto_parameters              = None


In [16]:
# Tokens que añaden los schemas de las tools a CADA llamada al modelo
print_tool_schema_tokens(tools)

  multiply            88 tokens
  tavily_search     1676 tokens
  fijo (uso de tools)    496 tokens
TOTAL schemas de tools: 2260 tokens, en CADA llamada al modelo


In [17]:
# Comprobación con una ejecución real: input de la primera llamada de "Hola, ¿qué puedes hacer?"
# (= system prompt + schemas de las tools + la pregunta; esa query no usó herramientas)
i_hola = next(k for k, q in enumerate(queries) if q["query"].startswith("Hola"))
print("input_tokens de la primera llamada:", first_call_input(results[i_hola]))

input_tokens de la primera llamada: 2559


### 4.3 ¿Cómo se generan los snippets?

Para ver de dónde sale cada snippet se repite la búsqueda pidiendo también la **página completa** (`include_raw_content=True`) y se compara:

- **literal**: qué % de los chunks aparece tal cual en la página (si es 100%, son extractos y no resúmenes).
- **pos.**: en qué parte de la página están (0 = inicio, 1 = final).
- **%pág**: qué parte de la página representa el snippet.

La lista `snippet_queries` es editable: añade los ejemplos vistos en clase (el de Alejandro, el del Mundial de Miguel Ángel).

In [18]:
snippet_queries = [
    "¿Quién ganó el Mundial de fútbol de Qatar 2022?",
    "¿Cuántos goles marcó Messi en el Mundial de Qatar 2022?",
    "¿Cuál es la última versión estable de LangGraph?",
    # añade aquí los ejemplos de clase
]

snippet_reports = {q: inspect_snippets(q) for q in snippet_queries}
for q, report in snippet_reports.items():
    print_snippet_report(q, report)
    print()

Query: ¿Quién ganó el Mundial de fútbol de Qatar 2022?
Términos de la query usados: ['2022', 'futbol', 'gano', 'mundial', 'qatar']
#    score  snippet   página   %pág  literal  pos.  cob.snip  cob.mejor  título
1     0.91     1240    10047    12%       0%     -       80%       100%  Quién ganó el Mundial 2022 y otros detal <-- hay un pasaje mejor
2     0.89     1357    15713     9%       0%     -       80%        80%  ¿Quién ganó y qué pasó en el Mundial 202
3     0.86     1202   150647     1%       0%     -       60%       100%  Final de la Copa Mundial de Fútbol de 20 <-- hay un pasaje mejor
4     0.81     1143     3118    37%       0%     -      100%       100%  Argentina campeón del mundo: la consagra
5     0.80      465    36414     1%     100%  0.92       40%        60%  ARGENTINA CAMPEÓN DEL MUNDO QATAR 2022 / <-- hay un pasaje mejor
literal = % de chunks que aparecen tal cual en la página | pos. = 0 inicio, 1 final
cob.snip / cob.mejor = % de términos de la query en el snippet 

In [19]:
# Lectura manual: los snippets que recibe el LLM para la primera query
demo_output = search_tool.invoke({"query": snippet_queries[0]})
for r in demo_output["results"][:3]:
    print(f"{r['title']}  (score {r['score']})")
    print(r["content"])
    print()

¿Quién ganó y qué pasó en el Mundial 2022? | DAZN News España  (score 0.89302075)
El Mundial de Qatar 2022,el primero celebrado en invierno, fue, sin duda, uno de los más especiales, espectaculares y sorprendentes de la historia. Marruecos alcanzó por primera vez las semifinales del Mundial y selecciones como Brasil, Inglaterra, Portugal o España, que no pudo pasar de octavos de final, se quedaron atrás en favor de países como Marruecos o Croacia.

Lee más | ¿Cuántos títulos tiene Leo Messi?

## ¿Quién ganó el Mundial 2022?

Argentina Mundial 2022Getty Images [...] Argentina Mundial 2022

Argentina se proclamó campeona del Mundial 2022 en Qatar después de superar a Francia en la final. El conjunto de Lionel Scaloni comenzó con mal pie la cita mundialista, pues cayó en la primera jornada de la fase de grupos ante Arabia Saudí, una derrota que les obligó a ganar a Polonia y México, algo que hicieron, para avanzar a octavos de final. [...] ### Balón de Oro al Mejor Jugador del Mundial de 

### 4.4 ¿Los snippets son realmente la parte relevante de la página?

Se mide con una aproximación **léxica**: qué porcentaje de los términos de la query aparece en el snippet (`cob.snip`) frente a la mejor ventana de 500 caracteres de la página (`cob.mejor`). Si la ventana mejor cubre más términos que el snippet, la tabla marca `<-- hay un pasaje mejor`.

Limitación: contar términos no mide significado. Por eso se complementa con la lectura manual de la celda anterior y con el juez del apartado siguiente.

### 4.5 ¿Cómo se genera el score y las respuestas son relevantes según el score?

Como Tavily no documenta el cálculo, se contrasta con una **valoración independiente**: un LLM puntúa cada snippet de 0 a 3 con un schema pydantic (`relevancia`, `explicacion`). Después se comprueba si el score ordena los resultados igual que el juez.

- **Coincidencia del orden**: fracción de pares de resultados en los que el de mayor relevancia tiene también mayor score (1.0 = coincide siempre, 0.5 = azar).
- **Score medio** de los resultados relevantes (>= 2) frente a los poco relevantes (<= 1): si apenas difiere, el score no sirve para filtrar.

Limitación: el juez es el mismo modelo (Haiku 4.5) y solo ve el snippet, no la página.

In [20]:
outputs_for_judge = {q: search_tool.invoke({"query": q}) for q in snippet_queries}
judged = {q: judge_results_relevance(q, out) for q, out in outputs_for_judge.items()}
print_score_vs_relevance(judged)


Query: ¿Quién ganó el Mundial de fútbol de Qatar 2022?
   score relev.  título / explicación
    0.89      3  ¿Quién ganó y qué pasó en el Mundial 2022? | DAZN 
                El fragmento responde directamente a la consulta, indicando claramente que Argentina ganó el Mundial de Qatar 
    0.86      3  Final de la Copa Mundial de Fútbol de 2022 - Wikip
                El fragmento responde directamente a la consulta, indicando claramente que Argentina ganó el Mundial de Qatar 
    0.81      3  Argentina campeón del mundo: la consagración de Qa
                El fragmento responde directamente a la consulta, indicando claramente que Argentina ganó el Mundial de Qatar 
    0.80      3  ARGENTINA CAMPEÓN DEL MUNDO QATAR 2022 // LA SCALO
                El fragmento responde directamente a la consulta, indicando claramente que Argentina ganó el Mundial de Qatar 
    0.80      3  Argentina, campeona del Mundial Qatar 2022 | Espec
                El fragmento responde directamente a la co

### 4.6 ¿El modelo usa realmente las respuestas o alucina?

Para cada query del Paso 1 que buscó en la web, un LLM comprueba si cada afirmación de la respuesta final aparece en los snippets que recibió el modelo (`respaldada`, `afirmaciones_sin_respaldo`). Además se listan los **números de la respuesta que no aparecen en las fuentes**, una señal barata de posibles alucinaciones.

In [21]:
evaluations = evaluate_groundedness(queries, results)
print_groundedness(evaluations)

Query                                                   Respaldada  Nº sin respaldo
¿Cuál es la última versión estable de LangGraph?        sí                        0
¿Qué tiempo hace hoy en Madrid?                         sí                        0
¿Cuál es el precio actual del Bitcoin?                  sí                        0
Resume las noticias de hoy sobre inteligencia arti...   sí                        0
¿Quién ganó la Champions League 2024-2025?              sí                        0
¿Quién ganó el Mundial de Clubes de la FIFA 2025?       sí                        0
¿Quién ganó el Tour de Francia 2025?                    sí                        0
Multiplica 12 por 34 y dime cuál es la última vers...   sí                        0

- ¿Cuál es el precio actual del Bitcoin?
    números que no están en las fuentes: ['74,500', '84,700']
    juez: Todas las afirmaciones factuales de la respuesta aparecen en las fuentes: los precios específicos de cada plataforma (CoinMarketCa

### 4.7 Conclusiones del Paso 2

*(Muestra: 3 queries y 15 resultados para snippets y score; 8 respuestas con búsqueda web para alucinaciones.)*

**Estructura de lo que devuelve Tavily**

- Claves: `query`, `follow_up_questions`, `answer`, `images`, `results`, `response_time`, `request_id`. Cada resultado trae `url`, `title`, `content`, `score`, `raw_content` e `id`.
- **El `content` (snippet) es el 82% del JSON.** `url`, `title`, `score` e `id` suman menos del 10%. `answer`, `images` y `raw_content` llegan vacíos porque la tool no los pide.
- En la query de LangGraph, el JSON de 7.557 caracteres pesa **2.341 tokens** en el ToolMessage.

**Lo que ve el LLM con `bind_tools()`**

- Los schemas suman **2.260 tokens en cada llamada**: `tavily_search` 1.676, `multiply` 88 y 496 fijos que Anthropic añade por usar tools.
- En la query "Hola", que no usa herramientas, la primera llamada tiene 2.559 tokens de input: **el 88% son schemas**.
- El LLM ve 9 parámetros de Tavily y solo `topic` está fijado. El 73% del JSON del schema (4.738 de 6.461 caracteres) son las descripciones de los parámetros (`end_date` 897 caracteres, `start_date` 759...).

**Snippets**

- Miden unos 1.100-1.500 caracteres: tres chunks unidos por `[...]`, como indica la documentación. Algunos de 120-140 caracteres (query de Messi) llegan sin página (`raw_content` vacío).
- **Literalidad:** de 12 resultados con página, solo 2 aparecen tal cual en `raw_content`, 3 parcialmente y 7 nada. Es una prueba conservadora (la página puede venir con otro formato), así que no refuta que sean extractos; solo que no se puede confirmar.
- **Relevancia:** hay un pasaje mejor en la página en 3 de los 12 casos. En la lectura manual aparece ruido: el snippet de Wikipedia sobre la final de Qatar 2022 son **referencias bibliográficas**, el de DAZN lleva texto de navegación, el de ESPN un aviso de edición y el de npm un banner sobre tokens y 2FA. Son pasajes parecidos a la consulta, pero no siempre la parte útil.

**Score**

- El juez puntuó 14 de 15 resultados como relevantes (>= 2). Score medio de relevantes **0.77** frente a **0.85** del único poco relevante: con esta muestra el score no discrimina.
- Coincidencia del orden con el juez: 75% (Messi) y 100% (LangGraph); la query de Qatar no tiene pares comparables.
- Hay resultados relevantes con score bajo (0.65 puntuado 3 y 0.54 puntuado 2). **Un umbral alto (por ejemplo 0.7) descartaría resultados útiles.**
- El score mide parecido con la consulta, no frescura ni verdad: en LangGraph aparecen la 1.2.7, la 1.2.16 y la 1.4.15 (esta última es el paquete de JavaScript). Además, la misma query dio resultados distintos entre ejecuciones.
- Limitaciones: muestra pequeña, juez benévolo y que es el mismo modelo, que solo ve el snippet.

**Alucinaciones**

- Con el evaluador original, 6 de 8 respuestas salieron respaldadas. Las 2 marcadas "NO" eran fallos del evaluador: en la mixta, el `408` viene de `multiply` y no de la web; en Bitcoin, el juez se contradecía y los números se comparaban como texto (`126,210.50` no coincide con `126210.5`). El script está corregido: ahora incluye los resultados y argumentos de las demás tools como fuente y compara los números por valor.
- **Pendiente:** volver a ejecutar la celda 4.6 con el script corregido y anotar aquí el recuento final.
- Hasta entonces: no hay evidencia de alucinaciones claras; el modelo se apoya en los resultados.

## 5. Resultados y conclusiones

**Paso 1: comportamiento**

- **14 de 15 queries** se comportaron como se esperaba. La única ✗ es "Champions 2024-2025": usó la web. La final fue en mayo de 2025, posterior al conocimiento fiable de Haiku 4.5 (febrero de 2025 según la documentación), así que buscar es defendible y la expectativa de la batería era discutible.
- **La fecha inyectada funciona:** "¿Qué día es hoy?" se respondió sin buscar, en una sola llamada.

**Paso 1: tokens y coste**

| Medida | Valor |
|---|---|
| Query sin herramientas | 2.583-3.071 tokens, de los que ~2.260 son schemas |
| ToolMessage de Tavily (8 queries con web) | media **2.874** tokens (mín. 2.341, máx. 4.474 con "tiempo en Madrid") |
| Peso del ToolMessage en el input total | media **35%** |
| Coste medio de los ToolMessages | **0,00287 USD** por búsqueda |
| `I2 - I1 - O1` frente a `count_tokens` | 2.341 frente a 2.329: diferencia de **12 tokens** (0,5%) |

**Dónde se va el coste de una búsqueda** (query de LangGraph: 7.553 de input, 365 de output, 0,0094 USD):

| Concepto | Tokens | Coste | % del coste |
|---|---|---|---|
| Schemas de las tools (2 llamadas x 2.260) | 4.520 | 0,00452 | **48%** |
| ToolMessage de Tavily | 2.341 | 0,00234 | 25% |
| Resto del input (system prompt, pregunta, tool_call) | 692 | 0,00069 | 7% |
| Output | 365 | 0,00183 | 19% |

**El mayor coste no es el resultado de Tavily, sino los schemas, que se pagan en cada llamada.** Es la hipótesis principal del Paso 3.

**Paso 2**

- Resumen en la sección 4.7: el snippet es el 82% del JSON, el score no discrimina con esta muestra, hay ruido en los snippets y no hay evidencia clara de alucinaciones (pendiente de re-ejecutar 4.6).

**Qué sugieren los datos para el Paso 3 (a medir, no a dar por hecho)**

1. **Schema mínimo:** una tool propia con solo `query` y el resto de parámetros fijados al crearla. El 73% del schema son descripciones de parámetros; el ahorro estaría en el orden de 1.500 tokens por llamada.
2. **ToolMessage más pequeño:** menos resultados, recortar el `content` o quitar el ruido (navegación, referencias).
3. **Filtro por score:** con umbral bajo o con los k mejores, porque uno alto descarta resultados útiles.
4. **Prompt caching:** el prefijo (system prompt + tools) es idéntico en cada llamada; hay que comprobar si supera el mínimo cacheable del modelo.
5. **Juez de relevancia dentro del agente:** añade una llamada por búsqueda; solo compensa si evita búsquedas inútiles o respuestas peores.